# fase_2 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_future'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to target database (db_future config): {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to target database (db_future config): dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
# Mapping tabel Hanif Fase 2: (Tabel Lama, Tabel Baru)
# Catatan: Kita sesuaikan sumber datanya agar tidak menyentuh tabel 'users' milik Cimut
hanif_tables_map = [
    ('hakakses', 'division_user'),
    ('kelurahan', 'kelurahan')
]

raw_data = {}

print("=== INSPEKSI SKEMA & RETRIEVAL DATA (FASE 2 - HANIF) ===\n")

for old_t, new_t in hanif_tables_map:
    try:
        print(f"📦 ANALISIS: {old_t} ➔ {new_t}")
        
        # Inspeksi metadata target (DB Baru)
        cursor_new.execute(f"DESCRIBE `{new_t}`")
        df_new_schema = pd.DataFrame(cursor_new.fetchall())

        # Tarik data mentah dari DB Lama
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()

        print(f"--- Skema Target ({new_t}) ---")
        display(df_new_schema[['Field', 'Type', 'Null', 'Key']])
        print(f"✅ {len(raw_data[old_t])} records loaded\n")

    except Exception as e:
        print(f"❌ ERROR di tabel {old_t}: {e}")

print("✓ Data mentah Fase 2 Hanif berhasil dimuat.")

=== INSPEKSI SKEMA & RETRIEVAL DATA (FASE 2 - HANIF) ===

📦 ANALISIS: hakakses ➔ division_user
--- Skema Target (division_user) ---


,Field,Type,Null,Key
0,id_division_user,varchar(15),NO,MUL
1,id_division,bigint(20) unsigned,NO,MUL
2,id_role,bigint(20) unsigned,NO,MUL
3,created_at,timestamp,YES,
4,updated_at,timestamp,YES,


✅ 6 records loaded

📦 ANALISIS: kelurahan ➔ kelurahan
--- Skema Target (kelurahan) ---


,Field,Type,Null,Key
0,id_kelurahan,bigint(20) unsigned,NO,PRI
1,id_kecamatan,bigint(20) unsigned,NO,MUL
2,nama_kelurahan,varchar(100),NO,
3,kode_pos,varchar(10),YES,


✅ 83473 records loaded

✓ Data mentah Fase 2 Hanif berhasil dimuat.


## 3. Transform Data (jika diperlukan)

In [4]:
now = datetime.now()
transformed_dfs = {}

# 1. division_user (Source: hakakses)
# Mapping: idusers -> id_user, iddivisi -> id_division
df_div_user = pd.DataFrame(raw_data['hakakses'])
df_div_user = df_div_user.rename(columns={'idusers': 'id_user', 'iddivisi': 'id_division'})
df_div_user['id_role'] = 0  # Default value
df_div_user['created_at'] = now
# Ambil kolom data murni saja (Abaikan auto-increment PK)
transformed_dfs['division_user'] = df_div_user[['id_user', 'id_division', 'id_role', 'created_at']]

# 2. kelurahan (Source: kelurahan lama)
# Mapping: idkecamatan -> id_kecamatan, nama -> nama_kelurahan
df_kel = pd.DataFrame(raw_data['kelurahan'])
df_kel = df_kel.rename(columns={'idkecamatan': 'id_kecamatan', 'nama': 'nama_kelurahan'})
df_kel['nama_kelurahan'] = df_kel['nama_kelurahan'].astype(str).str.strip()
# Abaikan id_kelurahan
transformed_dfs['kelurahan'] = df_kel[['id_kecamatan', 'nama_kelurahan']]

# 3. model_has_roles & model_has_permissions (Placeholder / Data Baru)
# Karena ini tabel relasi sistem baru (Spatie), kita siapkan DataFrame kosong 
# jika datanya belum tersedia di DB lama atau akan diisi lewat logic lain
transformed_dfs['model_has_roles'] = pd.DataFrame(columns=['role_id', 'model_type', 'model_id'])
transformed_dfs['model_has_permissions'] = pd.DataFrame(columns=['permission_id', 'model_type', 'model_id'])

print("✓ Transformasi Fase 2 Hanif selesai. Semua ID lama sudah dibuang.")

✓ Transformasi Fase 2 Hanif selesai. Semua ID lama sudah dibuang.


## 4. Insert ke Pickle untuk Insert Handler

In [5]:
# Nama file pkl khusus bagian Hanif di Fase 2
file_name = 'fase_2_hanif.pkl'

try:
    with open(file_name, 'wb') as f:
        pickle.dump(transformed_dfs, f)
    print(f"✅ Berhasil menyimpan {len(transformed_dfs)} tabel ke {file_name}")
    print("Siap diproses oleh insert_handler.ipynb!")
except Exception as e:
    print(f"❌ Gagal simpan pickle: {e}")

✅ Berhasil menyimpan 4 tabel ke fase_2_hanif.pkl
Siap diproses oleh insert_handler.ipynb!


## 5. Verifikasi Data

In [6]:
print("=== VERIFIKASI DATA FASE 2 HANIF SEBELUM EXPORT ===\n")

for table_name, df in transformed_dfs.items():
    print(f"📊 Tabel: {table_name}")
    print(f"   - Jumlah Record: {len(df)} baris")
    print(f"   - List Kolom: {df.columns.tolist()}")
    display(df.head(2))
    print("-" * 50)

=== VERIFIKASI DATA FASE 2 HANIF SEBELUM EXPORT ===

📊 Tabel: division_user
   - Jumlah Record: 6 baris
   - List Kolom: ['id_user', 'id_division', 'id_role', 'created_at']


,id_user,id_division,id_role,created_at
0,U00012,D00001,0,2026-06-21 19:28:03.106747
1,U00026,D00005,0,2026-06-21 19:28:03.106747


--------------------------------------------------
📊 Tabel: kelurahan
   - Jumlah Record: 83473 baris
   - List Kolom: ['id_kecamatan', 'nama_kelurahan']


,id_kecamatan,nama_kelurahan
0,110101,Keude Bakongan
1,110101,Ujong Mangki


--------------------------------------------------
📊 Tabel: model_has_roles
   - Jumlah Record: 0 baris
   - List Kolom: ['role_id', 'model_type', 'model_id']


,role_id,model_type,model_id


--------------------------------------------------
📊 Tabel: model_has_permissions
   - Jumlah Record: 0 baris
   - List Kolom: ['permission_id', 'model_type', 'model_id']


,permission_id,model_type,model_id


--------------------------------------------------


## 6. Return Hasil Migrasi

In [7]:
# Menghitung total records yang berhasil di-transform untuk semua tabel
total_processed = sum(len(df) for df in transformed_dfs.values())

migration_result = {
    'fase': 'fase_2',
    'script': 'script_hanif',
    'fase_num': 2,
    'status': 'ready_for_insert', # Status baru: Data siap di-insert
    'records_transformed': total_processed,
    'pickle_file': 'fase_2_hanif.pkl',
    'timestamp': datetime.now().isoformat(),
    'message': 'Transformasi Fase 2 Hanif selesai, file .pkl berhasil dibuat'
}

print("\n" + "="*60)
print("HASIL TRANSFORMASI - FASE 2 / SCRIPT_HANIF")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)


HASIL TRANSFORMASI - FASE 2 / SCRIPT_HANIF
{
  "fase": "fase_2",
  "script": "script_hanif",
  "fase_num": 2,
  "status": "ready_for_insert",
  "records_transformed": 83479,
  "pickle_file": "fase_2_hanif.pkl",
  "timestamp": "2026-06-21T19:28:03.353145",
  "message": "Transformasi Fase 2 Hanif selesai, file .pkl berhasil dibuat"
}


## 7. Close Connection

In [8]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")

✓ Database connections closed
